In [6]:
import pandas as pd
import numpy as np
from collections import Counter

# ----------- Sample Dataset (Play Tennis) -----------
data = pd.DataFrame({
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Mild','Hot','Cool','Mild','Mild','Mild','Hot','Mild'],
    'Humidity': ['High','High','High','High','Normal','Normal','Normal','High','Normal','Normal','Normal','High','Normal','High'],
    'Wind': ['Weak','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Strong'],
    'PlayTennis': ['No','No','Yes','Yes','Yes','No','Yes','No','Yes','Yes','Yes','Yes','Yes','No']
})

# ----------- Entropy Function -----------
def entropy(y):
    counts = Counter(y)
    probs = [count/len(y) for count in counts.values()]
    return -sum(p * np.log2(p) for p in probs if p > 0)

# ----------- Information Gain -----------
def info_gain(data, feature, target):
    total_entropy = entropy(data[target])
    values = data[feature].unique()
    
    weighted_entropy = 0
    for v in values:
        subset = data[data[feature] == v]
        weighted_entropy += (len(subset)/len(data)) * entropy(subset[target])
    
    return total_entropy - weighted_entropy

# ----------- ID3 Algorithm -----------
def id3(data, features, target):
    # If all target values same → return label
    if len(data[target].unique()) == 1:
        return data[target].iloc[0]
    
    # If no features left → return majority class
    if len(features) == 0:
        return data[target].mode()[0]
    
    # Select best feature (max information gain)
    gains = [info_gain(data, f, target) for f in features]
    best_feature = features[np.argmax(gains)]
    
    tree = {best_feature: {}}
    
    # Split dataset
    for value in data[best_feature].unique():
        subset = data[data[best_feature] == value]
        subtree = id3(
            subset,
            [f for f in features if f != best_feature],
            target
        )
        tree[best_feature][value] = subtree
    
    return tree

# ----------- Train Tree -----------
features = list(data.columns[:-1])
tree = id3(data, features, 'PlayTennis')

print("Decision Tree:")
print(tree)

# ----------- Prediction Function -----------
def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree
    
    root = next(iter(tree))
    value = sample[root]
    
    if value in tree[root]:
        return predict(tree[root][value], sample)
    else:
        return None

# ----------- Test Prediction -----------
test_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

print("\nTest Sample Prediction:", predict(tree, test_sample))

Decision Tree:
{'Outlook': {'Sunny': {'Temperature': {'Hot': 'No', 'Cool': 'Yes', 'Mild': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}

Test Sample Prediction: Yes
